# NumPy Broadcasting

## Fokus Bab

Broadcasting adalah salah satu fitur NumPy yang paling sering disalahpahami oleh pemula, namun sekaligus salah satu yang paling powerful. Bab ini membahas bagaimana NumPy memungkinkan operasi antar-array dengan shape yang berbeda, serta aturan pasti yang menentukan kapan operasi semacam itu diperbolehkan atau justru memicu error.

## Tujuan Pembelajaran

* Menjelaskan konsep broadcasting dan mengapa NumPy memerlukannya.
* Memprediksi apakah dua array dengan shape berbeda dapat dioperasikan bersama, hanya dengan membaca shape-nya.
* Menerapkan dua aturan resmi broadcasting NumPy untuk menentukan kompatibilitas shape.
* Membaca dan mendiagnosis error shape mismatch saat broadcasting gagal.
* Menerapkan broadcasting untuk studi kasus nyata: normalisasi dan standardisasi fitur, dua operasi yang sangat umum dalam tahap preprocessing Data Science.

## Goals (Target Output)

Mampu memprediksi apakah dua array dapat dioperasikan berdasarkan shape-nya, tanpa perlu trial-and-error — sekaligus mampu mendiagnosis dan memperbaiki error shape mismatch ketika terjadi. Kemampuan ini menjadi prasyarat langsung untuk memahami vectorization pada bab 5.

## Shape — Dimensi, Row, Column, dan Shape Compatibility

In [1]:
import pandas as pd
import numpy as np

In [4]:
matrix = np.array([[1, 2, 3], [4, 5, 6]])
print(matrix)
print("\nMatrix Shape")
print(matrix.shape)

[[1 2 3]
 [4 5 6]]

Matrix Shape
(2, 3)


* Angka pertama pada tuple shape, yaitu 2, merepresentasikan jumlah baris (row).
* Angka kedua, yaitu 3, merepresentasikan jumlah kolom (column).
* Shape compatibility adalah istilah untuk menyatakan apakah dua array dengan shape berbeda dapat "dipasangkan" dalam sebuah operasi elemen-per-elemen. Inilah inti dari broadcasting: menentukan kompatibilitas tersebut.

## Broadcasting Scalar

Broadcasting adalah mekanisme NumPy untuk membuat array dengan shape yang berbeda menjadi kompatibel secara konseptual sehingga operasi element-wise dapat dilakukan, tanpa harus benar-benar menyalin data yang di-broadcast.

Bentuk broadcasting paling sederhana adalah ketika sebuah array dioperasikan dengan sebuah skalar (nilai tunggal)

In [5]:
arr = np.array([1, 2, 3])
hasil = arr + 10
print(hasil)

[11 12 13]


* Secara matematis, arr memiliki shape (3,) sedangkan 10 adalah skalar tanpa dimensi (shape ()).
* NumPy "membayangkan" skalar 10 tersebut diperluas (stretched) menjadi array [10, 10, 10] agar shape-nya cocok dengan arr, sehingga operasi elemen-per-elemen bisa dilakukan: [1+10, 2+10, 3+10] → [11, 12, 13].
* Perlu ditekankan: "stretching" ini bersifat konseptual, bukan literal. NumPy tidak benar-benar menggandakan nilai 10 di memori — ini murni optimasi internal agar broadcasting tetap efisien secara memori dan komputasi, sekalipun secara logika terlihat seolah nilai tersebut disalin ke setiap posisi.

## Broadcasting 1D → 2D

In [8]:
matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])

vektor = np.array([10, 20, 30])
hasil = matrix + vektor

print(hasil)

[[11 22 33]
 [14 25 36]
 [17 28 39]]


Penjelasan kode di atas:

* matrix memiliki shape (3, 3), sedangkan vector memiliki shape (3,).
* Karena jumlah elemen pada vector (3) cocok dengan jumlah kolom pada matrix (3), NumPy akan "meregangkan" vector secara virtual agar seolah-olah menjadi array (3, 3), di mana setiap barisnya adalah salinan dari vector yang sama.
* Hasil akhirnya: setiap baris pada matrix ditambahkan dengan vector yang sama persis.

Berikut ilustrasi visual dari proses tersebut:

![Ilustrasi broadcasting](../assets/figures/numpy_broadcasting_matrix_vector.png)

Kotak dengan garis putus-putus pada diagram di atas merepresentasikan sifat "virtual" dari proses stretching — secara konsep vector tersebut seolah digandakan ke tiga baris, tetapi NumPy tidak benar-benar mengalokasikan memori tambahan untuk itu.

## Broadcasting Rules

* **Aturan 1 — Penyelarasan dimensi**

  * Jika jumlah dimensi berbeda, tambahkan dimensi `1` pada **sisi kiri** shape yang dimensinya lebih sedikit.
  * Contoh:

    ```text
    (3,) → (1, 3)
    (2, 3) → (2, 3)
    ```

* **Aturan 2 — Kompatibilitas setiap dimensi**

  * Bandingkan dimensi dari **kanan ke kiri**.
  * Dua dimensi kompatibel jika:

    * ukurannya **sama**, atau
    * salah satunya bernilai **1**.
  * Jika tidak memenuhi keduanya → **broadcasting gagal**.

* **Inti:**

  > **Broadcasting membuat shape yang berbeda menjadi kompatibel untuk operasi element-wise berdasarkan aturan dimensi di atas.**


```text
Dimensi dibandingkan dari KANAN ke KIRI
        ↓
Untuk setiap pasangan dimensi:
  kompatibel jika → SAMA
             atau → salah satu = 1
        ↓
Jika tidak kompatibel di dimensi manapun → ValueError
```

Contoh membaca shape compatibilty

```python
Contoh 1 — kompatibel
A: (3, 3)
B:    (3,)  → dibaca sebagai (1, 3) setelah padding
Hasil: (3, 3)

Contoh 2 — kompatibel
A: (5, 1)
B: (1, 4)
Hasil: (5, 4)

Contoh 3 — TIDAK kompatibel
 A: (3, 4)
 B: (3, 3)
 Dimensi terakhir: 4 vs 3 → tidak sama dan tidak ada yang bernilai 1 → error
```

Penjelasan untuk Contoh 2:

* Kedua array sama-sama "kecil" pada satu sisi ((5,1) dan (1,4)), namun keduanya tetap kompatibel karena pada setiap posisi dimensi, salah satu nilainya adalah 1.
* Hasilnya adalah kedua array saling meregang ke arah yang berlawanan, menghasilkan shape (5, 4) — sebuah matrix yang lebih besar dari kedua array aslinya. Pola ini sering dipakai untuk membuat tabel hasil kombinasi (outer operation) tanpa loop bersarang.

## Shape Mismatch — Debugging

Jika kedua array tidak memenuhi kedua aturan broadcasting akan value error

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])   # shape (2, 3)
b = np.array([1, 2])                    # shape (2,)

a + b

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 

Penjelasan mengapa error ini muncul:

* `a` memiliki `shape (2, 3)`, sedangkan b memiliki `shape (2,)` yang setelah padding menjadi (1, 2).
* Dibandingkan dari kanan: dimensi terakhir `a` adalah `3`, dimensi terakhir `b` adalah `2` — keduanya tidak sama, dan tidak ada yang bernilai 1. Aturan broadcasting gagal terpenuhi, sehingga NumPy menolak operasi tersebut.

Langkah debugging yang direkomendasikan ketika menjumpai error semacam ini:

* Cetak shape kedua array yang terlibat `(print(a.shape, b.shape))` — jangan menebak, langsung verifikasi.
* Bandingkan dari kanan ke kiri, dimensi demi dimensi, sesuai aturan pada subbab 4.5.
* Jika array memang dimaksudkan untuk beroperasi sepanjang axis tertentu, gunakan `.reshape()` atau `np.newaxis` untuk menyisipkan `dim`

In [10]:
b_reshaped = b.reshape(2, 1)   # shape (2, 1)
a + b_reshaped                  # sekarang kompatibel → hasil shape (2, 3)

array([[2, 3, 4],
       [6, 7, 8]])

Baris kode di atas menyisipkan dimensi baru pada b sehingga shape-nya berubah dari (2,) menjadi (2, 1) — kini dimensi terakhirnya 1, yang menurut Aturan 2 selalu kompatibel dengan dimensi berapa pun pada a.

## Studi Kasus: Normalisasi & Standardisasi Fitur

### Standardisasi (mengubah data agar memiliki mean 0 dan standar deviasi 1):

In [2]:
import numpy as np
data = np.array([[170, 65], [180, 80], [160, 55]])  # shape (3, 2)

mean = data.mean(axis=0)
std = data.std(axis=0)

data_standard = (data - mean) / std
print(data_standard)

[[ 0.         -0.16222142]
 [ 1.22474487  1.29777137]
 [-1.22474487 -1.13554995]]


Penjelasan kode di atas:

* `data.mean(axis=0)` menghasilkan shape `(2,)` — satu nilai rata-rata untuk setiap kolom (konsep axis akan dibahas mendalam di bab 6).
* Operasi data - mean melibatkan `array (3, 2)` dan `(2,)` — broadcasting berlaku persis seperti pada subbab 4.4, di mana mean diregangkan ke setiap baris.
* Pembagian dengan std mengikuti mekanisme broadcasting yang sama, sehingga seluruh proses standardisasi dapat ditulis dalam satu baris tanpa loop sama sekali.

### Normalisasi Min-Max (mengubah data ke rentang 0–1):

In [4]:
data_min = data.min(axis=0)
data_max = data.max(axis=0)

data_normal = (data - data_min) / (data_max - data_min)
print(data_normal)

[[0.5 0.4]
 [1.  1. ]
 [0.  0. ]]


Pola ini — mengurangi array (n, m) dengan hasil agregasi ber-shape (m,) — adalah salah satu penerapan broadcasting yang paling sering muncul di seluruh workflow Data Science, mulai dari EDA hingga feature engineering sebelum modeling.